<a href="https://colab.research.google.com/github/salamlakhan7/sprint-03-deep-learning-nlp-transformers/blob/main/Notebooks/RNN_stock_predict.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Recurrent Neural Networks**

**Concept:** Every model so far — ANN, CNN - looks at input all at once and has zero memory of order. An RNN is different: it processes input one step at a time, and carries forward a hidden state - a running summary of everything it's seen so far - from one step to the next.

**Real-life example:** Reading a sentence word by word. By the time you reach the word "it" in "The dog chased the ball because it was fun," you understand "it" refers to the ball - but only because you remembered the earlier words. If someone showed you just the word "it" alone (like ANN treating inputs independently), you'd have no idea what it means. RNN's hidden state is exactly that running memory - carrying context forward, step by step.

# **(Stock Price Context)**

**Concept:** An RNN processes a sequence one step at a time, carrying forward a hidden state - a running summary of everything it's seen so far. For stock prices, this means: instead of looking at today's price in isolation, the model remembers the trend from the past several days when making its next prediction.

**Real-life example:** Think of a trader glancing at a stock chart. They don't just look at today's closing price in isolation - they mentally carry forward "it's been climbing for 5 days" or "it just had a sharp drop." That running mental summary, updated day by day, is exactly what an RNN's hidden state does mathematically.

**The task we'll build:** Given the last N days of closing prices, predict the next day's closing price. This is a classic sequence-to-one regression problem - perfect for isolating pure RNN mechanics without any NLP complexity getting in the way, which is exactly why you picked this option.

## **Step 1 - Getting Stock Data**

**Concept:** We need a real, ordered sequence of numbers. yfinance pulls historical stock prices directly, free, no API key needed.

In [1]:
!pip install yfinance --quiet

import yfinance as yf
import pandas as pd

data = yf.download("AAPL", start="2015-01-01", end="2024-01-01")
close_prices = data['Close'].values
print(close_prices.shape)
print(close_prices[:5])

/tmp/ipykernel_2011/73786880.py:6: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download("AAPL", start="2015-01-01", end="2024-01-01")
[*********************100%***********************]  1 of 1 completed

(2264, 1)
[[24.17175293]
 [23.49079895]
 [23.49301147]
 [23.82243729]
 [24.73774147]]


## **Step 2 - Building Sequences (Last N Days → Next Day)**

**Concept:** Raw prices are just a flat list right now - [24.17, 23.49, 23.49, ...]. An RNN needs windows: "here are the last 30 days, predict day 31." We slide a window across the whole price history to generate many such training examples from one continuous list.


 Think of it like a weather forecaster's training data - they don't learn from a single day in isolation, they learn from many overlapping "last week → tomorrow" patterns pulled from years of history. Same sliding-window idea here.

In [2]:
import numpy as np

def create_sequences(data, window_size):
    X, y = [], []
    for i in range(len(data) - window_size):
        X.append(data[i:i+window_size])      # last `window_size` days
        y.append(data[i+window_size])         # the very next day
    return np.array(X), np.array(y)

window_size = 30
X, y = create_sequences(close_prices, window_size)

print("X shape:", X.shape)   # (num_samples, window_size, 1)
print("y shape:", y.shape)   # (num_samples, 1)

X shape: (2234, 30, 1)
y shape: (2234, 1)


Why window_size = 30: roughly a trading month - enough history to give the RNN meaningful trend context, without going so long that training gets slow. This is a hyperparameter you can experiment with later.

shapes - 2,234 overlapping 30-day windows, each predicting the very next day.

## **Step 3 - Scaling + Time-Aware Train/Val Split**

 **part 1 - Scaling:** Raw prices range from ~20 to ~200+ across this history. Neural networks train much better when inputs are in a small, consistent range (like 0-1) - large raw numbers can cause unstable gradients. This is the same idea as Normalize in your ANN/CNN work, just a different technique (min-max scaling) suited to continuous numeric data.

**part 2 - Why we can't shuffl**e: With MNIST, shuffling was fine - each image is independent. Here, time order matters. If we randomly shuffled before splitting, some future days could end up in training while past days end up in validation - the model would essentially be "peeking into the future," making your results meaningless. We must split by keeping validation data strictly after training data on the timeline.

 Imagine training a weather forecaster using days from all across the year randomly mixed, then testing on random days too - some test days might come from earlier than train days. That forecaster could accidentally learn from a "future" heatwave to predict a "past" one. Real deployment never works that way - you always predict forward in time from what you know now, so validation should mimic that.

In [3]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
close_scaled = scaler.fit_transform(close_prices)   # scale to 0-1

X, y = create_sequences(close_scaled, window_size)

# Time-aware split: first 80% for training, last 20% for validation — no shuffling
split_idx = int(len(X) * 0.8)
X_train, X_val = X[:split_idx], X[split_idx:]
y_train, y_val = y[:split_idx], y[split_idx:]

print("Train:", X_train.shape, "Val:", X_val.shape)

Train: (1787, 30, 1) Val: (447, 30, 1)


## **Step 4 - Converting to PyTorch Tensors & DataLoaders**

**Concept:** Same idea as MNIST's DataLoader, just for numeric sequences instead of images. PyTorch needs data as Tensor objects, and batching still helps training efficiency here too.

In [4]:
import torch
from torch.utils.data import TensorDataset, DataLoader

X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)
X_val_t   = torch.tensor(X_val, dtype=torch.float32)
y_val_t   = torch.tensor(y_val, dtype=torch.float32)

train_dataset = TensorDataset(X_train_t, y_train_t)
val_dataset   = TensorDataset(X_val_t, y_val_t)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=False)  # no shuffle — time order matters
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## **Step 5 - Building the RNN Architecture**

 **This is the actual RNN layer -** nn.RNN in PyTorch. Unlike CNN's Conv2d which slides a filter over space, nn.RNN slides through time steps, updating a hidden state at each step and passing it forward.

**Think of the trader again,** reading a stock chart day by day. At each new day, they update their "gut feeling" about the trend (hidden state) based on today's price combined with their gut feeling from yesterday. By day 30, that gut feeling has absorbed all 30 days of context - and that's what makes the final prediction.

In [7]:
import torch.nn as nn

class SimpleRNN(nn.Module):
    def __init__(self, input_size=1, hidden_size=32, num_layers=1):
        super().__init__()
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)   # final layer: hidden state -> single price prediction

    def forward(self, x):
        out, hidden = self.rnn(x)       # out: hidden state at EVERY time step
        last_step = out[:, -1, :]       # we only care about the FINAL time step's hidden state
        prediction = self.fc(last_step)
        return prediction

model = SimpleRNN().to(device)


**Breaking down the key parameters:**

| Parameter | Meaning |
|---|---|
| `input_size=1` | Each time step has 1 number (a single closing price) |
| `hidden_size=32` | Size of the "memory" vector carried between time steps — bigger = more capacity to remember patterns |
| `num_layers=1` | One RNN layer (can stack more later, like CNN's conv blocks) |
| `batch_first=True` | Tells PyTorch our data shape is (batch, time_steps, features) — matches our `(32, 30, 1)` batches |


**Why `out[:, -1, :]`:** the RNN produces a hidden state at *every* one of the 30 time steps, but we only want the model's understanding *after* seeing all 30 days - that's the very last time step, which has absorbed the full sequence's context.

Run this cell to confirm no errors, then Step 6 will set up the loss function and optimizer - this time using **MSELoss** instead of CrossEntropyLoss, since we're predicting a continuous price, not a class.

## **Step 6 - Loss Function & Optimizer**

**Same role as before - loss measures wrongness, optimizer corrects it** - but a different loss function this time. ANN/CNN predicted categories (which digit, 0-9), so we used CrossEntropyLoss. Here we're predicting a continuous number (tomorrow's price), so we need MSELoss (Mean Squared Error) - it measures how far off a numeric prediction is, not how wrong a category guess is.

**Guessing a category is like a multiple-choice question** - you're either right or wrong. Predicting a price is like guessing someone's age - being off by 1 year is a small mistake, being off by 30 years is a huge one. MSE captures that "how far off" distance, squared to penalize big misses more heavily than small ones.

In [9]:
import torch.optim as optim

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

## **Step 7 - The Training Loop**

Same Input → Model → Prediction → Loss → Learning cycle as ANN and CNN. Only real differences: data comes from train_loader as (sequences, targets) instead of (images, labels), and we're using MSELoss now instead of CrossEntropyLoss.

In [10]:
def train_one_epoch():
    model.train()
    total_loss = 0
    for sequences, targets in train_loader:
        sequences, targets = sequences.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(sequences)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(train_loader)

## **Step 8 - Validation Loop**

**Same as always - no learning**, just checking how well the model predicts prices it hasn't trained on. Structurally identical to ANN/CNN's validation loop, just with the new loss function and data shape.

In [11]:
def validate():
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for sequences, targets in val_loader:
            sequences, targets = sequences.to(device), targets.to(device)
            outputs = model(sequences)
            loss = criterion(outputs, targets)
            total_loss += loss.item()
    return total_loss / len(val_loader)

## **Step 9 - Running It & Tracking Loss**

**loop through epochs, train, validate,** print both losses to watch how well the model is learning.

In [12]:
epochs = 20
for epoch in range(epochs):
    train_loss = train_one_epoch()
    val_loss = validate()
    print(f"Epoch {epoch+1}: train_loss={train_loss:.6f}, val_loss={val_loss:.6f}")

Epoch 1: train_loss=0.005831, val_loss=0.017882
Epoch 2: train_loss=0.106866, val_loss=0.140355
Epoch 3: train_loss=0.053125, val_loss=0.087071
Epoch 4: train_loss=0.021568, val_loss=0.008012
Epoch 5: train_loss=0.006324, val_loss=0.002011
Epoch 6: train_loss=0.001064, val_loss=0.002895
Epoch 7: train_loss=0.001000, val_loss=0.002368
Epoch 8: train_loss=0.000817, val_loss=0.002058
Epoch 9: train_loss=0.000933, val_loss=0.001837
Epoch 10: train_loss=0.000931, val_loss=0.001613
Epoch 11: train_loss=0.000969, val_loss=0.001443
Epoch 12: train_loss=0.000962, val_loss=0.001296
Epoch 13: train_loss=0.000954, val_loss=0.001179
Epoch 14: train_loss=0.000931, val_loss=0.001080
Epoch 15: train_loss=0.000908, val_loss=0.000996
Epoch 16: train_loss=0.000882, val_loss=0.000922
Epoch 17: train_loss=0.000858, val_loss=0.000858
Epoch 18: train_loss=0.000835, val_loss=0.000801
Epoch 19: train_loss=0.000815, val_loss=0.000751
Epoch 20: train_loss=0.000796, val_loss=0.000706


In [13]:
import numpy as np

model.eval()
with torch.no_grad():
    preds = model(X_val_t.to(device)).cpu().numpy()
    actuals = y_val_t.numpy()

# Reverse the 0-1 scaling back to real prices
preds_real = scaler.inverse_transform(preds)
actuals_real = scaler.inverse_transform(actuals)

mae = np.mean(np.abs(preds_real - actuals_real))
print(f"Average prediction error: ${mae:.2f}")

Average prediction error: $3.88


In [14]:
naive_preds = actuals_real[:-1]   # yesterday's actual price
naive_actuals = actuals_real[1:]  # today's actual price

naive_mae = np.mean(np.abs(naive_preds - naive_actuals))
print(f"Naive baseline (predict no change) error: ${naive_mae:.2f}")

Naive baseline (predict no change) error: $2.07


In [15]:
print(f"Validation period price range: ${actuals_real.min():.2f} - ${actuals_real.max():.2f}")
print(f"Average price in validation period: ${actuals_real.mean():.2f}")

Validation period price range: $122.83 - $195.72
Average price in validation period: $160.52


In [16]:
print(f"Validation period price range: ${actuals_real.min():.2f} - ${actuals_real.max():.2f}")
print(f"Average price in validation period: ${actuals_real.mean():.2f}")

Validation period price range: $122.83 - $195.72
Average price in validation period: $160.52
